# 12 — Gate 4: the ensemble runner (spec v0.12 §7, §8, §10)

Solves every frozen formulation in `spec/manifest.csv`, serial, **fully resumable** (each artifact
skipped when its output exists). Per formulation, into `analyses/y2y/runs/<formulation_id>/`:

| artifact | what | config |
|---|---|---|
| `kbest/` | engine run, k-best pool | Gurobi binary, opt_gap 1e-4, portfolio 50 @ 5% |
| `twin/` | engine run, LP twin | Gurobi **proportion** (the v0.10 twin ruling) |
| `anchor.tif` + `mga_g05.tif` | certified anchor + 50 MGA members | `mga_core`, g=5%, k=50 |

ssp245 formulations solve on the 245 macrorefugia realization (layer path patched before ingest;
recorded in each `formulation_meta.json`). Reference formulation: anchor/MGA exist from Gate 2b; kbest/twin
are manifest pointers to the Gate-2 record. ~45 min/formulation ⇒ **~10 h for the 13 open formulations**;
**live internet throughout** (WLS). Kernel `R (y2y)`.

In [1]:
ANALYSIS <- "y2y"
PROJ <- normalizePath(getwd())
while (!file.exists(file.path(PROJ, "config.py"))) {
  parent <- dirname(PROJ)
  if (identical(parent, PROJ)) stop("config.py not found above getwd() -- open from inside the repo")
  PROJ <- parent
}
setwd(PROJ)
source(file.path(PROJ, "prioritizr_core.R"))
source(file.path(PROJ, "mga_core.R"))

mpath <- pr_refresh_manifest(PROJ, ANALYSIS)

MAN <- read.csv(file.path(PROJ, "analyses/y2y/spec/manifest.csv"), stringsAsFactors = FALSE)
stopifnot(nrow(MAN) == 14)
# verify the freeze hash before solving against it
dig <- strsplit(readLines(file.path(PROJ, "analyses/y2y/spec/manifest_freeze.sha256"))[1], "  ")[[1]][1]
stopifnot("manifest.csv does not match its freeze hash -- STOP" =
            identical(unname(tools::sha256sum(file.path(PROJ, "analyses/y2y/spec/manifest.csv"))[[1]]), dig))
cat(sprintf("manifest verified against freeze hash %s...\n", substr(dig, 1, 16)))
RUNS <- file.path(PROJ, "analyses/y2y/runs")
REAL245 <- "input_data/aligned_stack/climate_realizations/macrorefugia_245_2071_2100.tif"

manifest refreshed from config.py (analysis=y2y)
manifest verified against freeze hash d45668bbb8773da3...


In [2]:
# ---- two ingested base contexts, built ONCE (one per climate level) ------------------------
ctx585 <- pr_setup(mpath, PROJ)
ctx585 <- modifyList(ctx585, pr_ingest(ctx585))
ctx585 <- modifyList(ctx585, pr_planning_units(ctx585))

ctx245 <- pr_setup(mpath, PROJ)
ctx245$layers$path[ctx245$layers$name == "climate_type_macrorefugia"] <- REAL245
ctx245 <- modifyList(ctx245, pr_ingest(ctx245))
ctx245 <- modifyList(ctx245, pr_planning_units(ctx245))
cat("base contexts ready (585 canonical; 245 with the realization layer patched)\n")

base_for <- function(row) if (grepl("^ssp245", row$climate_level)) ctx245 else ctx585

prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y
ingested 48 features (8 continuous + 40 EFG) + cost + PA mask | grid 1286 x 3312 @ 1000 m
normalized 48 features to total=100000 each (scale-invariant conditioning)
planning units: 1,272,914 cells | budget = 30% = 381,874 cells
locked-in [pa_mask]: 191,029 cells (15.0% of window) -- fits within budget
prioritizr 8.1.0 | terra 1.9.34 | analysis=y2y | solver=highs (single solution)
objective=min_shortfall | budget=30% target=100% | opt_gap=0.10 | time_limit=43200s
resolution: 1000 m (agg factor 1) | decisions=proportion
roi: mode=full | lock_in=pa_mask
penalties: connectivity=0 | boundary=0 | neighbor=0
outputs -> output_data/iter6_y2y
ingested 48 feature

In [3]:
# ---- DRY PLAN (no solves): the per-formulation worklist -------------------------------------------
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]
  cd <- file.path(RUNS, row$formulation_id)
  kb <- if (nzchar(row$kbest_ref)) "ref" else if (file.exists(file.path(cd, "kbest/run_summary.json"))) "done" else "TODO"
  tw <- if (nzchar(row$twin_ref))  "ref" else if (file.exists(file.path(cd, "twin/run_summary.json")))  "done" else "TODO"
  mg <- if (file.exists(file.path(cd, "mga_g05.tif"))) "done" else "TODO"
  cat(sprintf("%-22s %-9s kbest:%-5s twin:%-5s mga:%-5s\n",
              row$formulation_id, sub("_2071_2100", "", row$climate_level), kb, tw, mg))
}

s0_ssp585_theta5       ssp585    kbest:ref   twin:ref   mga:done 
s1_ssp585_theta5       ssp585    kbest:TODO  twin:TODO  mga:TODO 
s2_ssp585_theta5       ssp585    kbest:TODO  twin:TODO  mga:TODO 
s3_ssp585_theta5       ssp585    kbest:TODO  twin:TODO  mga:TODO 
s4_ssp585_theta3       ssp585    kbest:TODO  twin:TODO  mga:TODO 
s5_ssp585_theta5       ssp585    kbest:TODO  twin:TODO  mga:TODO 
s0_ssp245_theta5       ssp245    kbest:TODO  twin:TODO  mga:TODO 
s1_ssp245_theta5       ssp245    kbest:TODO  twin:TODO  mga:TODO 
s2_ssp245_theta5       ssp245    kbest:TODO  twin:TODO  mga:TODO 
s3_ssp245_theta5       ssp245    kbest:TODO  twin:TODO  mga:TODO 
s4_ssp245_theta3       ssp245    kbest:TODO  twin:TODO  mga:TODO 
s5_ssp245_theta5       ssp245    kbest:TODO  twin:TODO  mga:TODO 
s1x_ssp585_theta3      ssp585    kbest:TODO  twin:TODO  mga:TODO 
s3x_ssp585_theta3      ssp585    kbest:TODO  twin:TODO  mga:TODO 


In [4]:
# ---- runner helpers ------------------------------------------------------------------------
form_wt <- function(row) list(w = jsonlite::fromJSON(row$weight_vector),
                              t = jsonlite::fromJSON(row$target_vector))

run_engine_artifact <- function(row, artifact, ov) {
  cd_rel <- file.path("analyses/y2y/runs", row$formulation_id)
  done <- file.path(PROJ, cd_rel, artifact, "run_summary.json")
  if (file.exists(done)) { cat(sprintf("   %s/%s exists -- skipped\n", row$formulation_id, artifact)); return(invisible(NULL)) }
  wt <- form_wt(row)
  actx <- do.call(pr_override, c(list(base_for(row),
      targets                    = wt$t,
      feature_weight_multipliers = wt$w,
      results_dir                = cd_rel,
      results_subdir             = artifact), ov))
  actx <- modifyList(actx, pr_weights(actx))
  actx <- modifyList(actx, pr_targets(actx))
  actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  sv <- pr_solve(actx); actx$s <- sv$s; actx$timing <- sv$timing
  actx$n_sol <- sv$n_sol; actx$sol_attrs <- sv$sol_attrs
  actx <- modifyList(actx, pr_summaries(actx))
  pr_write_outputs(actx)
  invisible(NULL)
}

run_mga_artifact <- function(row) {
  cd <- file.path(RUNS, row$formulation_id)
  if (file.exists(file.path(cd, "mga_g05.tif"))) {
    cat(sprintf("   %s/mga exists -- skipped\n", row$formulation_id)); return(invisible(NULL)) }
  wt <- form_wt(row)
  actx <- pr_override(base_for(row),
      targets = wt$t, feature_weight_multipliers = wt$w,
      results_dir = file.path("analyses/y2y/runs", row$formulation_id),
      results_subdir = "mga_build",
      solver = "gurobi", decision_type = "binary", opt_gap = 1e-4, portfolio_n = 1)
  actx <- modifyList(actx, pr_weights(actx))
  actx <- modifyList(actx, pr_targets(actx))
  actx <- modifyList(actx, pr_penalty_matrices(actx))
  bp <- pr_build_problem(actx); actx$p <- bp$p; actx$solve_params <- bp$solve_params
  cm <- mga_compile(actx)
  anchor <- mga_anchor(cm, opt_gap = row$opt_gap)
  gen <- mga_generate(cm, anchor, g = row$band_gap_g, k = row$k_requested)
  mga_write(gen, cm, actx$cost, cd, "g05")
  r <- terra::rast(actx$cost); v <- rep(NA_integer_, terra::ncell(r))
  v[cm$pu_index] <- as.integer(anchor$x); terra::values(r) <- v
  terra::writeRaster(r, file.path(cd, "anchor.tif"), overwrite = TRUE, datatype = "INT1U",
                     NAflag = 255, gdal = c("COMPRESS=DEFLATE", "TILED=YES"))
  jsonlite::write_json(list(
    formulation_id = row$formulation_id, estimator = row$estimator, verdict_rule = row$verdict_rule,
    anchor_objective = anchor$z, anchor_bound = anchor$bound, anchor_gap = anchor$gap,
    anchor_runtime_s = anchor$runtime,
    macrorefugia_path = if (grepl("^ssp245", row$climate_level)) REAL245
                        else "input_data/aligned_stack/climate_type_macrorefugia.tif",
    weight_vector = form_wt(row)$w, target_vector = form_wt(row)$t,
    k = row$k_requested, g = row$band_gap_g,
    created_utc = format(Sys.time(), tz = "UTC")),
    file.path(cd, "formulation_meta.json"), auto_unbox = TRUE, pretty = TRUE, digits = 10)
  invisible(NULL)
}

In [5]:
# ---- THE LOOP: serial over the frozen formulations (resumable anywhere) --------------------------
t_batch <- proc.time()[["elapsed"]]
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]
  cat(sprintf("\n===================== %s (%d/%d) =====================\n", row$formulation_id, i, nrow(MAN)))
  if (!nzchar(row$kbest_ref)) {
    run_engine_artifact(row, "kbest", list(solver = "gurobi", decision_type = "binary",
                                           opt_gap = row$opt_gap, portfolio_n = row$k_requested,
                                           portfolio_gap = row$band_gap_g))
  } else cat(sprintf("   kbest -> %s (Gate-2 record)\n", row$kbest_ref))
  if (!nzchar(row$twin_ref)) {
    run_engine_artifact(row, "twin", list(solver = "gurobi", decision_type = "proportion",
                                          opt_gap = row$opt_gap, portfolio_n = 1))
  } else cat(sprintf("   twin  -> %s (Gate-2 record, HiGHS exact)\n", row$twin_ref))
  # reference formulation: Gate 2b wrote gate2b_meta.json before the naming settled --
  # derive the standard meta file once so 13 reads every formulation uniformly
  cd <- file.path(RUNS, row$formulation_id)
  g2b <- file.path(cd, "gate2b_meta.json")
  fmeta <- file.path(cd, "formulation_meta.json")
  if (!file.exists(fmeta) && file.exists(g2b)) {
    m <- jsonlite::read_json(g2b)
    m$formulation_id <- row$formulation_id
    jsonlite::write_json(m, fmeta, auto_unbox = TRUE, pretty = TRUE, digits = 10)
    cat("   formulation_meta.json derived from gate2b_meta.json\n")
  }
  run_mga_artifact(row)
  cat(sprintf("== %s done | batch elapsed %.1f h\n", row$formulation_id,
              (proc.time()[["elapsed"]] - t_batch) / 3600))
}
cat("\nENSEMBLE COMPLETE -- next: analyses/y2y/13_gate4_analysis.ipynb\n")


===================== s0_ssp585_theta5 (1/14) =====================
   kbest -> output_data/iter9_y2y_s0_pool (Gate-2 record)
   twin  -> output_data/iter9_y2y_s0_lp (Gate-2 record, HiGHS exact)
   formulation_meta.json derived from gate2b_meta.json
   s0_ssp585_theta5/mga exists -- skipped
== s0_ssp585_theta5 done | batch elapsed 0.0 h

===================== s1_ssp585_theta5 (2/14) =====================
  override targets          -> irrecoverable_carbon_m_soc=0.332
  override feature_weight_multipliers -> climate_type_macrorefugia=3.09075, transboundary_connectivity=0.472299, climate_corridors=0.826581, irrecoverable_carbon_m_soc=0.327869, irrecoverable_carbon_biomass=0.140112, aoh_richness_birds=0.937497, aoh_richness_mammals=1.20489
  override results_dir      -> analyses/y2y/runs/s1_ssp585_theta5
  override results_subdir   -> kbest
  override solver           -> gurobi
  override decision_type    -> binary
  override opt_gap          -> 1e-04
  override portfolio_n      -> 50
  

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.332 and 1)
││└•weights:    continuous values (between 0.025 and 3.090749)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   gap portfolio (`number_solutions` = 50, `pool_gap` = 0.05)
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
Set parameter PoolSolutions to value 50
Set parameter PoolSearchMode to value 2
Set parameter PoolGap to value 0.05
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10
PoolSolutions  50
PoolSearchMode  2
PoolGap  0.05

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 49 rows, 1272962 columns and 16912559 nonzeros (Min)
Model fingerprint: 0xefa316bb
Model has 48 linear objective coefficients
Variable types: 48 co

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.332 and 1)
││└•weights:    continuous values (between 0.025 and 3.090749)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   proportion decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 49 rows, 1272962 columns and 16912559 nonzeros (Min)
Model fingerprint: 0x1deb224c
Model has 48 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e-04, 1e+05]
  Objective range  [3e-02, 3e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [3e+04, 4e+05]

Presolve removed 38 rows a

...

Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
Set parameter PoolSolutions to value 50
Set parameter PoolSearchMode to value 2
Set parameter PoolGap to value 0.05
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10
PoolSolutions  50
PoolSearchMode  2
PoolGap  0.05

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 49 rows, 1272962 columns and 16912559 nonzeros (Min)
Model fingerprint: 0xe8a42d13
Model has 48 linear objective coefficients
Variable types: 48 co

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.552 and 1)
││└•weights:    continuous values (between 0.025 and 2.743654)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   proportion decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



Set parameter WLSAccessID
Set parameter WLSSecret
Set parameter LicenseID to value 2853457
Set parameter TimeLimit to value 43200
Set parameter NumericFocus to value 2
Set parameter Presolve to value 2
Set parameter Threads to value 10
WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Gurobi Optimizer version 13.0.2 build v13.0.2rc1 (mac64[arm] - Darwin 25.6.0 25G83)

CPU model: Apple M4
Thread count: 10 physical cores, 10 logical processors, using up to 10 threads

Non-default parameters:
TimeLimit  43200
NumericFocus  2
Presolve  2
Threads  10

WLS license 2853457 - registered to Yellowstone to Yukon Conservation Initiative
Optimize a model with 49 rows, 1272962 columns and 16912559 nonzeros (Min)
Model fingerprint: 0x9ef0ea6c
Model has 48 linear objective coefficients
Coefficient statistics:
  Matrix range     [1e-04, 1e+05]
  Objective range  [3e-02, 3e+00]
  Bounds range     [1e+00, 1e+00]
  RHS range        [6e+04, 4e+05]

Presolve removed 38 rows a

A conservation problem (<ConservationProblem>)
├•data
│├•features:    "human_modification", "transboundary_connectivity", … (48 total)
│└•planning units:
│ ├•data:       <SpatRaster> (1272914 total)
│ ├•costs:      constant values (all equal to 1)
│ ├•extent:     -2206000, 273000, -920000, 3585000 (xmin, ymin, xmax, ymax)
│ └•CRS:        North_America_Albers_Equal_Area_Conic (projected)
├•formulation
│├•objective:   minimum shortfall objective (`budget` = 381874.2)
│├•penalties:   none specified
│├•features:
││├•targets:    relative targets (between 0.552 and 1)
││└•weights:    continuous values (between 0.025 and 2.743654)
│├•constraints: 
││└•1:          locked in constraints (191029 planning units)
│└•decisions:   binary decision
└•optimization
 ├•portfolio:   default portfolio
 └•solver:      gurobi solver (`gap` = 0.0001, `time_limit` = 43200, `first_feasible` = FALSE, …)
# ℹ Use `summary(...)` to see complete formulation.



compiled: 1272962 cols (1272914 pu + 48 aux) x 49 rows | 191029 locked pu | modelsense min
anchor: objective 5.619003 (bound 5.619003, gap 0.00e+00) | 381,874 selected | 14 s
band wall appended: obj0 . x <= 5.899953  (g = 0.05 on z* = 5.619003)
g=0.05 iter 01/50: band 5.899952 (+5.00% of z*) OK | ham(anchor) 371,282 | 51 s
g=0.05 iter 02/50: band 5.899950 (+5.00% of z*) OK | ham(anchor) 330,896 | 48 s
g=0.05 iter 03/50: band 5.899953 (+5.00% of z*) OK | ham(anchor) 272,700 | 50 s
g=0.05 iter 04/50: band 5.899953 (+5.00% of z*) OK | ham(anchor) 208,136 | 53 s
g=0.05 iter 05/50: band 5.899953 (+5.00% of z*) OK | ham(anchor) 213,318 | 53 s
g=0.05 iter 06/50: band 5.899952 (+5.00% of z*) OK | ham(anchor) 314,576 | 56 s
g=0.05 iter 07/50: band 5.899951 (+5.00% of z*) OK | ham(anchor) 274,436 | 61 s
g=0.05 iter 08/50: band 5.899952 (+5.00% of z*) OK | ham(anchor) 232,724 | 55 s
g=0.05 iter 09/50: band 5.899953 (+5.00% of z*) OK | ham(anchor) 211,356 | 52 s
g=0.05 iter 10/50: band 5.899954 (+

In [8]:
# ---- timing + integrity summary ------------------------------------------------------------
for (i in seq_len(nrow(MAN))) {
  row <- MAN[i, ]; cd <- file.path(RUNS, row$formulation_id)
  meta_f <- file.path(cd, "formulation_meta.json")
  kb_f <- if (nzchar(row$kbest_ref)) file.path(PROJ, row$kbest_ref, "run_summary.json")
          else file.path(cd, "kbest/run_summary.json")
  tw_f <- if (nzchar(row$twin_ref)) file.path(PROJ, row$twin_ref, "run_summary.json")
          else file.path(cd, "twin/run_summary.json")
  if (!file.exists(meta_f) || !file.exists(kb_f) || !file.exists(tw_f)) {
    cat(sprintf("%-22s INCOMPLETE\n", row$formulation_id)); next }
  m <- jsonlite::read_json(meta_f); kb <- jsonlite::read_json(kb_f); tw <- jsonlite::read_json(tw_f)
  tw_obj <- tryCatch(as.numeric(unlist(tw$solver_provenance$objective))[1], error = function(e) NA)
  ok <- if (is.na(tw_obj)) "?" else if (tw_obj <= m$anchor_objective + 1e-6) "OK" else "VIOLATED"
  cat(sprintf("%-22s anchor %.6f (gap %.0e, %4.0fs) | twin %.6f [LP<=MILP %s] | kbest %d sol %5.0fs\n",
              row$formulation_id, m$anchor_objective, m$anchor_gap, m$anchor_runtime_s,
              ifelse(is.na(tw_obj), NaN, tw_obj), ok, kb$n_alternatives, kb$solve_seconds))
}

s0_ssp585_theta5       anchor 5.362800 (gap 0e+00,    8s) | twin 5.362800 [LP<=MILP OK] | kbest 50 sol  1799s
s1_ssp585_theta5       anchor 5.188700 (gap 1e-04,   53s) | twin 5.188400 [LP<=MILP OK] | kbest 50 sol  1822s
s2_ssp585_theta5       anchor 5.565500 (gap 0e+00,   52s) | twin 5.565400 [LP<=MILP OK] | kbest 50 sol   849s
s3_ssp585_theta5       anchor 5.520800 (gap 0e+00,   50s) | twin 5.520600 [LP<=MILP OK] | kbest 38 sol 43210s
s4_ssp585_theta3       anchor 5.014300 (gap 6e-06,   49s) | twin 5.014300 [LP<=MILP OK] | kbest 50 sol  1227s
s5_ssp585_theta5       anchor 11.632000 (gap 0e+00,   45s) | twin 11.631600 [LP<=MILP OK] | kbest 50 sol   768s
s0_ssp245_theta5       anchor 5.319100 (gap 1e-06,   44s) | twin 5.319100 [LP<=MILP OK] | kbest 50 sol  1379s
s1_ssp245_theta5       anchor 5.095300 (gap 7e-06,   55s) | twin 5.095200 [LP<=MILP OK] | kbest 50 sol   940s
s2_ssp245_theta5       anchor 5.548900 (gap 0e+00,   58s) | twin 5.548700 [LP<=MILP OK] | kbest 50 sol   549s
s3_ssp24

In [9]:
# ---- OPTIONAL: HiGHS spot-check twin (2nd cell; reference already has one) -----------------
# Flip to TRUE and run overnight if desired (worst observed HiGHS case: 109 min).
RUN_HIGHS_SPOTCHECK <- FALSE
if (RUN_HIGHS_SPOTCHECK) {
  row <- MAN[MAN$formulation_id == "s4_ssp585_theta3", ]
  run_engine_artifact(row, "twin_highs", list(solver = "highs", decision_type = "proportion",
                                              portfolio_n = 1))
}